# 04 — Train Pose Correction Model

Train a dual-output model: **phase prediction** + **correct landmark generation**.

## Phase 1 approach: Correct-Form Only

We train exclusively on correct-form videos. The model learns:
> "At phase X of exercise Y, the body should look like Z."

At inference, the user's actual (possibly incorrect) landmarks go in, and the model
outputs what correct form looks like at the detected phase. The difference = correction signal.

## Model architecture
```
Input (99 landmarks + exercise one-hot)
  -> Shared encoder: Dense(128) -> Dense(64) -> Dense(32)
     |                    |
     v                    v
  Phase head           Correction head
  Dense(16)->Dense(1)  Concat(shared, phase)->Dense(64)->Dense(128)->Dense(99)
     |                    |
  Phase (0-1)          Corrected landmarks (99 floats)
```

## Inputs
```
data/labeled/*_labeled.npz  — landmarks + phases
```

## Outputs
```
models/pose_correction.keras  — trained model
models/test_fixtures.npz      — test data for TFLite validation
```

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from src.dataset import load_labeled_data, prepare_dataset, split_by_video
from src.model import build_pose_correction_model, train_model, evaluate_model, get_model_summary
from src.normalization import INPUT_DIM, LANDMARK_FLAT, NUM_EXERCISES

print(f'TensorFlow version: {tf.__version__}')
print(f'GPUs available: {len(tf.config.list_physical_devices("GPU"))}')
print(f'Model input dim: {INPUT_DIM} (99 landmarks + {NUM_EXERCISES} exercise one-hot)')
print(f'Landmark output dim: {LANDMARK_FLAT}')

%matplotlib inline

## Load and prepare dataset

In [ ]:
raw_data = load_labeled_data('../data/labeled')

print(f'Loaded {len(raw_data["video_names"])} video(s):')
for name, exercise in zip(raw_data['video_names'], raw_data['exercise_names']):
    idx = raw_data['video_names'].index(name)
    n_frames = len(raw_data['landmarks_33'][idx])
    print(f'  {exercise:>8s} | {name} ({n_frames} frames)')

In [ ]:
# Prepare dataset with augmentation (~6x expansion) and masking
dataset = prepare_dataset(raw_data, augment=True, mask_fraction=0.3)

print(f'\nDataset prepared:')
print(f'  Total samples: {len(dataset["inputs"]):,}')
print(f'  Input shape: {dataset["inputs"].shape}  (99 landmarks + exercise one-hot)')
print(f'  Target shape: {dataset["landmark_targets"].shape}  (99 correct landmarks)')
print(f'  Phase range: [{dataset["phase_targets"].min():.3f}, {dataset["phase_targets"].max():.3f}]')

In [ ]:
# Split by video (not frame) to prevent data leakage
train, val, test = split_by_video(dataset, train_frac=0.8, val_frac=0.1)

print(f'Train: {len(train["inputs"]):,} samples')
print(f'Val:   {len(val["inputs"]):,} samples')
print(f'Test:  {len(test["inputs"]):,} samples')

if len(test['inputs']) == 0:
    print('\nWARNING: No test samples. With only 1-2 videos, all data goes to train/val.')
    print('This is OK for Phase 1 — we validate on val set instead.')

## Build and inspect model

In [ ]:
model = build_pose_correction_model()
print(get_model_summary(model))
print(f'\nTotal parameters: {model.count_params():,}')

## Train

- EarlyStopping with patience=15 (restores best weights)
- ReduceLROnPlateau halves LR if val_loss plateaus for 7 epochs
- Max 100 epochs (usually converges in 30-50)

In [ ]:
history = train_model(
    model, train, val,
    epochs=100,
    batch_size=64,
    patience=15,
)

## Training curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Val')
axes[0].set_title('Total Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['phase_loss'], label='Train')
axes[1].plot(history.history['val_phase_loss'], label='Val')
axes[1].set_title('Phase Loss (MSE)')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(history.history['correction_loss'], label='Train')
axes[2].plot(history.history['val_correction_loss'], label='Val')
axes[2].set_title('Correction Loss (MSE)')
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Evaluate

Uses test set if available, otherwise falls back to validation set.

In [ ]:
eval_data = test if len(test['inputs']) > 0 else val
eval_label = 'test' if len(test['inputs']) > 0 else 'val'

metrics = evaluate_model(model, eval_data)

print(f'Evaluation on {eval_label} set:')
print(f'  Phase MAE: {metrics["phase_mae"]:.4f}  (target: < 0.05)')
print(f'  Correction MAE: {metrics["correction_mae"]:.4f}  (target: < 0.001)')
print(f'\nPer-joint MAE (top 10 worst):')
sorted_joints = sorted(metrics['per_joint_mae'].items(), key=lambda x: -x[1])
for joint, mae in sorted_joints[:10]:
    bar = '#' * int(mae * 500)
    print(f'  {joint:>15s}: {mae:.4f} {bar}')

In [ ]:
# Visualize predictions vs targets
n_vis = min(100, len(eval_data['inputs']))
phase_pred, correction_pred = model.predict(eval_data['inputs'][:n_vis], verbose=0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Phase prediction scatter
ax1.scatter(eval_data['phase_targets'][:n_vis], phase_pred.flatten()[:n_vis], alpha=0.5, s=10)
ax1.plot([0, 1], [0, 1], 'r--', alpha=0.5)
ax1.set_xlabel('True Phase')
ax1.set_ylabel('Predicted Phase')
ax1.set_title('Phase Prediction (closer to red line = better)')
ax1.grid(True, alpha=0.3)

# Correction error histogram
errors = np.abs(correction_pred[:n_vis] - eval_data['landmark_targets'][:n_vis]).flatten()
ax2.hist(errors, bins=50, alpha=0.7, color='steelblue')
ax2.axvline(errors.mean(), color='red', linestyle='--', label=f'Mean: {errors.mean():.4f}')
ax2.set_xlabel('Absolute Error')
ax2.set_ylabel('Count')
ax2.set_title('Landmark Correction Error Distribution')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Save model and test fixtures

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

model.save('../models/pose_correction.keras')
print('Model saved to models/pose_correction.keras')

# Save test fixtures for TFLite validation (notebook 05)
n_fix = min(10, len(eval_data['inputs']))
fix_phase, fix_correction = model.predict(eval_data['inputs'][:n_fix], verbose=0)

np.savez(
    '../models/test_fixtures.npz',
    inputs=eval_data['inputs'][:n_fix],
    phase_expected=fix_phase[:n_fix],
    correction_expected=fix_correction[:n_fix],
)
print('Test fixtures saved to models/test_fixtures.npz')

---
**Next:** Run `05_export_tflite.ipynb` to convert to TFLite for mobile deployment.